**Homework 1**
- Vincenzo D'Angelo        M63001595
- Giorgio Di Costanzo        M63001579

# Installazione Apache Spark

In [1]:
''' DECOMMENTA PER INSTALLAZIONE SENZA DRIVE

!apt-get install openjdk-8-jdk-headless -qq > /dev/null
!wget -q https://archive.apache.org/dist/spark/spark-3.4.4/spark-3.4.4-bin-hadoop3.tgz
!tar xf spark-3.4.4-bin-hadoop3.tgz
!pip install -q findspark
'''

In [1]:
''' DECOMMENTA PER INSTALLAZIONE CON DRIVE
'''

from google.colab import drive
drive.mount('/content/drive')

# Download e installazione di Apache Spark 3.4.4, insieme alle dipedenze
!apt-get install openjdk-8-jdk-headless -qq > /dev/null
!pip install -q findspark

path_dataset = "/content/drive/MyDrive/Big Data/dimensions_clinicalTrials.csv"
!tar xzf "/content/drive/MyDrive/Big Data/spark-3.4.4-bin-hadoop3.tgz" -C /content/

Mounted at /content/drive


In [2]:
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
os.environ["SPARK_HOME"] = "spark-3.4.4-bin-hadoop3"

In [3]:
import findspark
findspark.init()
from pyspark.context import SparkContext
sc = SparkContext.getOrCreate()

assert  "3." in sc.version, "Verify that the cluster Spark's version is 3.x"

In [4]:
from pyspark.sql.functions import count
from pyspark.sql.functions import when, col, avg, desc, explode, split, trim, size, format_number, regexp_replace, regexp_extract
from pyspark.sql.functions import row_number, collect_list, concat_ws
from pyspark.sql.functions import datediff, to_date, mean, stddev, round as F_round
from pyspark.sql import functions as F

In [5]:
print("Spark version:", sc.version)

Spark version: 3.4.4


# Caricamento Dataset

In [6]:
from pyspark.sql import SparkSession
spark = SparkSession(sc)
#sc = SparkContext.getOrCreate()
df = spark.read.option("header", "true") \
               .option("inferSchema", "true") \
               .option("multiLine", "true") \
               .option("escape", '"') \
               .option("quote", '"') \
               .option("mode","PERMISSIVE") \
               .csv(path_dataset)

In [7]:
df.show(10)
df.printSchema()
row = df.count()
print("\nRighe totali:", row)

from pyspark.sql.functions import col, sum, when

print("\nValori nulli per colonna:")
df.select([
    sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in df.columns
]).show()


+----+--------------+--------------------+--------------------+-----------------+--------------------+----------+----------+----------+---------------+-------+--------------+--------------------+--------------------+--------------------+----------------------+--------------------+------+--------------------+------------------+----------------------+----------------------+----------------------------+-----------------------------+-------------------------------+---------------------+------------+--------------+--------------------+-------------------------+--------------------+--------------------------------+--------------------+--------------------+-------------------+--------------------+--------------------+---------------+
|Rank|      Trial ID|               Title|         Brief title|          Acronym|            Abstract|Start date|Start Year|  End Date|Completion Year|  Phase|    Study Type|        Study Design|          Conditions|  Recruitment Status|Number of Participants|   

# Analisi 1: Studi per cui si prevede una durata maggiore

In [10]:
from pyspark.sql.functions import to_date, months_between, year, datediff, min, max, round, when

df_dates = df.withColumn("Start Date", to_date(col("Start Date"), "dd/MM/yyyy")) \
            .withColumn("End Date", to_date(col("End Date"), "dd/MM/yyyy")) \
            .filter(col("Start Date").isNotNull())\
            .filter(col("End Date").isNotNull())\
            .filter(year(col("End Date"))>2025)\
            .groupBy("Trial ID")\
            .agg(
                min(col("Start Date")).alias("Start Date"),
                max(col("End Date")).alias("End Date")
            )\
            .withColumn("Expected Duration (years)", round(months_between(col("End Date"), col("Start Date"))/12, 2))\
            .orderBy("Expected Duration (years)", ascending = False)\
            .show(truncate=False)

+-----------+----------+----------+-------------------------+
|Trial ID   |Start Date|End Date  |Expected Duration (years)|
+-----------+----------+----------+-------------------------+
|NCT03553420|2017-12-16|2068-01-01|50.04                    |
|NCT01574053|2012-07-01|2062-01-01|49.5                     |
|NCT02440763|2005-07-01|2050-07-01|45.0                     |
|NCT02699736|1994-01-01|2030-12-01|36.92                    |
|NCT04725422|2018-08-01|2050-08-01|32.0                     |
|NCT03807687|2016-12-01|2046-12-01|30.0                     |
|NCT05845801|2022-12-07|2052-09-30|29.81                    |
|NCT05925699|2023-09-01|2052-09-01|29.0                     |
|NCT05522374|2012-06-14|2040-12-01|28.47                    |
|NCT00330447|2005-08-01|2032-12-01|27.33                    |
|NCT02832245|2001-03-01|2027-12-01|26.75                    |
|NCT01007474|2004-01-01|2030-01-01|26.0                     |
|NCT04036357|2011-11-01|2036-12-31|25.16                    |
|NCT0498

# Analisi 2: Studi completati in minor tempo

In [11]:
df_dates2 = df.withColumn("Start Date", to_date(col("Start Date"), "dd/MM/yyyy")) \
            .withColumn("End Date", to_date(col("End Date"), "dd/MM/yyyy")) \
            .filter(col("Start Date").isNotNull())\
            .filter(col("End Date").isNotNull())\
            .groupBy("Trial ID")\
            .agg(
                min("Start Date").alias("Start Date"),
                max("End Date").alias("End Date"),
            )\
            .withColumn("Duration (days)", round(datediff(col("End Date"), col("Start Date")), 2))\
            .orderBy("Duration (days)", ascending = True)\
            .select("Trial ID", "Start Date","End Date", "Duration (days)")\
            .show(truncate=False)

+------------+----------+----------+---------------+
|Trial ID    |Start Date|End Date  |Duration (days)|
+------------+----------+----------+---------------+
|NCT03832673 |2019-05-22|2019-05-22|0              |
|NCT04930172 |2022-10-30|2022-10-30|0              |
|NCT00054392 |2001-09-01|2001-09-01|0              |
|NCT03962608 |2021-01-31|2021-01-31|0              |
|NCT04391218 |2021-05-01|2021-05-01|0              |
|NCT05412706 |2023-09-04|2023-09-04|0              |
|EUPAS50139  |2022-12-12|2022-12-12|0              |
|EUPAS48420  |2022-07-29|2022-07-29|0              |
|NCT01501370 |2012-01-01|2012-01-01|0              |
|DRKS00028535|2021-05-27|2021-05-31|4              |
|NCT05197868 |2022-01-20|2022-01-26|6              |
|NCT04698694 |2020-12-09|2020-12-21|12             |
|NCT04247490 |2019-12-17|2019-12-31|14             |
|NCT04897152 |2021-08-15|2021-08-31|16             |
|NCT05430503 |2022-05-24|2022-06-15|22             |
|NCT05986032 |2023-09-01|2023-09-24|23        

# Analisi 3: Condizioni mediche più studiate in Italia nell'ultimo anno

In [12]:
df.withColumn("Conditions", split(col("Conditions"), "; "))\
  .withColumn("Conditions", explode(col("Conditions")))\
  .withColumn("Country of Sponsor/Collaborator", split(col("Country of Sponsor/Collaborator"), "; "))\
  .withColumn("Country of Sponsor/Collaborator", explode(col("Country of Sponsor/Collaborator")))\
  .filter(col("Conditions").isNotNull())\
  .filter(year(col("End Date")) > 2025)\
  .filter(col("Country of Sponsor/Collaborator")=="Italy")\
  .groupBy("Conditions") \
  .count() \
  .withColumnRenamed("count", "Number of Medical Conditions") \
  .orderBy("Number of Medical Conditions", ascending = False)\
  .show(truncate = False)

+----------------------------------------------+----------------------------+
|Conditions                                    |Number of Medical Conditions|
+----------------------------------------------+----------------------------+
|Follicular Lymphoma                           |2060                        |
|Breast Cancer                                 |1427                        |
|Melanoma                                      |699                         |
|Uveitis                                       |693                         |
|Still Disease                                 |684                         |
|Primary Sclerosing Cholangitis                |684                         |
|Scleritis                                     |684                         |
|PFAPA Syndrome                                |684                         |
|Schnitzler Syndrome                           |684                         |
|Vexas Syndrome                                |684             

# Analisi 4: Campi di ricerca con il più alto interesse mediatico

In [13]:
df.withColumn("Fields of Research (ANZSRC 2020)", split(col("Fields of Research (ANZSRC 2020)"), "; "))\
  .withColumn("Fields of Research (ANZSRC 2020)", explode(col("Fields of Research (ANZSRC 2020)")))\
  .filter(col("Fields of Research (ANZSRC 2020)").isNotNull())\
  .groupBy("Fields of Research (ANZSRC 2020)") \
  .agg(round(avg("Altmetric Attention Score"),2).alias("Avg Altmetric Attention Score")) \
  .orderBy("Avg Altmetric Attention Score", ascending = False)\
  .show(truncate = False)

+--------------------------------------------+-----------------------------+
|Fields of Research (ANZSRC 2020)            |Avg Altmetric Attention Score|
+--------------------------------------------+-----------------------------+
|5202 Biological Psychology                  |142.09                       |
|52 Psychology                               |135.77                       |
|3213 Paediatrics                            |122.75                       |
|4003 Biomedical Engineering                 |122.64                       |
|3101 Biochemistry and Cell Biology          |121.35                       |
|3212 Ophthalmology and Optometry            |117.22                       |
|40 Engineering                              |103.29                       |
|3203 Dentistry                              |91.2                         |
|4806 Private Law and Civil Obligations      |82.0                         |
|4102 Ecological Applications                |77.0                         |

# Analisi 5: AHC con la più alta percentuale di studi su anziani e minori

In [14]:
from pyspark.sql.functions import col, count, sum, when, round, regexp_extract

# Funzione di conversione ad anni
def convert_to_years(value_col, unit_col):
    return when(unit_col.contains("Year"), value_col) \
        .when(unit_col.contains("Month"), value_col / 12) \
        .when(unit_col.contains("Week"), value_col / 52) \
        .when(unit_col.contains("Day"), value_col / 365) \
        .otherwise(None)

# Rimuove solo righe in cui entrambi i valori sono N/A o None
df_filtered = df.filter(~col("Age").rlike("(?i)^(N/A|None)\s*-\s*(N/A|None)$"))

# Estrai gli estremi degli intervalli
df_extracted = df_filtered.withColumn("Age Min Value",
        when(col("Age").rlike("(?i)^(N/A|None)"), None)
        .otherwise(regexp_extract(col("Age"), r"^(\d+)", 1).cast("float"))) \
    .withColumn("Age Max Value",
        when(col("Age").rlike("(?i)-(?:\s*)?(N/A|None)$"), None)
        .otherwise(regexp_extract(col("Age"), r"-\s*(\d+)", 1).cast("float"))) \
    .withColumn("Age Min Unit",
        when(col("Age").rlike("(?i)^(N/A|None)"), None)
        .otherwise(regexp_extract(col("Age"), r"^(\d+)\s*(\w+)", 2))) \
    .withColumn("Age Max Unit",
        when(col("Age").rlike("(?i)-(?:\s*)?(N/A|None)$"), None)
        .otherwise(regexp_extract(col("Age"), r"-\s*(\d+)\s*(\w+)", 2)))

# Creazione categorie
df_cleaned=df_extracted.withColumn("Age Min Years", convert_to_years(col("Age Min Value"), col("Age Min Unit"))) \
    .withColumn("Age Max Years", convert_to_years(col("Age Max Value"), col("Age Max Unit"))) \
    .withColumn("Age Group", when(col("Age Max Years") < 18, "Pediatric")
                            .when(col("Age Min Years") >= 65, "Senior")
                            .when((col("Age Min Years") < 18) & (col("Age Max Years") > 18), "Mixed")
                            .otherwise("Adult"))

# Aggregazione per AHC con conteggi per ciascuna categoria
df_summary = df_cleaned.groupBy("AHC").agg(
    count("*").alias("Total Studies"),
    sum(when(col("Age Group") == "Pediatric", 1).otherwise(0)).alias("Pediatric Studies"),
    sum(when(col("Age Group") == "Adult", 1).otherwise(0)).alias("Adult Studies"),
    sum(when(col("Age Group") == "Senior", 1).otherwise(0)).alias("Senior Studies"),
    sum(when(col("Age Group") == "Mixed", 1).otherwise(0)).alias("Mixed Studies")
).withColumn("Pediatric %", round(col("Pediatric Studies") / col("Total Studies") * 100, 2)) \
 .withColumn("Adult %", round(col("Adult Studies") / col("Total Studies") * 100, 2)) \
 .withColumn("Senior %", round(col("Senior Studies") / col("Total Studies") * 100, 2)) \
 .withColumn("Mixed %", round(col("Mixed Studies") / col("Total Studies") * 100, 2))

# Visualizza i top AHC per Pediatric
print("AHC con la più alta percentuale di studi Pediatrici:")
df_summary.select("AHC", "Pediatric %") \
    .orderBy(col("Pediatric %").desc_nulls_last()) \
    .show(truncate=False)

# Visualizza i top AHC per Senior
print("AHC con la più alta percentuale di studi per Anziani (Senior):")
df_summary.select("AHC", "Senior %") \
    .orderBy(col("Senior %").desc_nulls_last()) \
    .show(truncate=False)

# Visualizza alcuni record Pediatric
print("Esempi di studi Pediatric:")
df_cleaned.filter(col("Age Group") == "Pediatric") \
    .select("AHC", "Age", "Age Min Years", "Age Max Years", "Age Group") \
    .show(10, truncate=False)

# Visualizza alcuni record Senior
print("Esempi di studi Senior:")
df_cleaned.filter(col("Age Group") == "Senior") \
    .select("AHC", "Age", "Age Min Years", "Age Max Years", "Age Group") \
    .show(10, truncate=False)

# Visualizza alcuni record Adult
print("Esempi di studi Adult:")
df_cleaned.filter(col("Age Group") == "Adult") \
    .select("AHC", "Age", "Age Min Years", "Age Max Years", "Age Group") \
    .show(10, truncate=False)

AHC con la più alta percentuale di studi Pediatrici:
+-----------------------+-----------+
|AHC                    |Pediatric %|
+-----------------------+-----------+
|IRCCS_BURLOGAROFOLO    |50.0       |
|IRCCS_MEYER            |41.59      |
|IRCCS_GASLINI          |36.02      |
|AOUSSN_GMARTINO        |14.4       |
|IRCCS_CAGRANDA         |8.95       |
|AOUSSN_CAGLIARI        |8.57       |
|AOU_SANGIOVANNIDIDIO   |8.33       |
|AOUSSN_FEDERICOII      |7.81       |
|IRCCS_BESTA            |7.78       |
|AOU_PADOVA             |7.46       |
|IRCCS_BONINOPULEJO     |7.14       |
|AOUSSN_VITTORIOEMANUELE|6.48       |
|AOU_NOVARAGALLIATE     |6.25       |
|AOU_VERONA             |5.98       |
|IRCCS_INRCA            |5.56       |
|IRCCS_SANGERARDO       |4.23       |
|IRCCS_SANMATTEO        |3.82       |
|AOUSSN_VANVITELLI      |3.45       |
|AOUSSN_UMBERTOI        |3.44       |
|IRCCS_SANTORSOLA       |3.23       |
+-----------------------+-----------+
only showing top 20 rows

AHC con l

# Analisi 6: Rapporto tra durata attesa e numero di partecipanti per ogni fase

In [15]:
# Creazione colonna durata
df.withColumn("Duration",datediff(to_date(col("End Date"), "yyyy-MM-dd"), to_date(col("Start date"), "yyyy-MM-dd")))\
    .filter((df.Phase.isNotNull()) &(df["Number of Participants"].isNotNull()))\
    .filter(col("Duration").isNotNull())\
    .groupBy("Phase")\
    .agg(
      round(avg("Duration"), 2).alias("Media_Durata_Prevista"),
      round(avg("Number of Participants"), 2).alias("Media_Partecipanti"),
      round(avg("Duration") / avg("Number of Participants"), 2).alias("Giorni_per_Partecipante")
    )\
    .select("Phase", "Media_Durata_Prevista", "Media_Partecipanti", "Giorni_per_Partecipante")\
    .orderBy("Phase")\
    .show(truncate=False)


+--------------------------+---------------------+------------------+-----------------------+
|Phase                     |Media_Durata_Prevista|Media_Partecipanti|Giorni_per_Partecipante|
+--------------------------+---------------------+------------------+-----------------------+
|Phase 1                   |1510.56              |98.59             |15.32                  |
|Phase 1/2                 |1958.93              |164.82            |11.88                  |
|Phase 2                   |1701.0               |149.41            |11.38                  |
|Phase 2/3                 |1748.98              |733.27            |2.39                   |
|Phase 3                   |2039.25              |886.57            |2.3                    |
|Phase 3/4                 |876.0                |315.0             |2.78                   |
|Phase 4                   |1413.27              |505.49            |2.8                    |
|Post Authorisation Studies|1023.27              |57036.82  

# Analisi 7: Durata stimata media e numero medio di partecipanti per tipologia di studio

In [16]:
df.withColumn("Duration", datediff(to_date(col("End Date"), "yyyy-MM-dd"), to_date(col("Start date"), "yyyy-MM-dd")))\
  .filter((df["Study Type"].isNotNull()) & (df["Number of Participants"].isNotNull()))\
  .filter(col("Duration").isNotNull())\
  .groupBy("Study Type") \
  .agg(
    round(avg("Duration"),2).alias("Media Durata Attesa"),
    round(avg("Number of Participants"),2).alias("Media Partecipanti")
  ) \
  .orderBy("Media Durata Attesa")\
  .show()

+-------------------+-------------------+------------------+
|         Study Type|Media Durata Attesa|Media Partecipanti|
+-------------------+-------------------+------------------+
| Non-interventional|              436.0|             185.0|
|Active surveillance|              487.0|          115000.0|
|      Observational|            1671.93|           2646.27|
|     Interventional|            1801.84|            649.38|
|                CCT|             2648.0|             520.0|
|              Other|             3531.0|             202.0|
+-------------------+-------------------+------------------+



#Analisi 8: Durata stimata media, numero medio di partecipanti e attenzione mediatica media per ogni categoria HRCS

In [17]:
df.withColumn("Duration", datediff(to_date(col("End Date"), "yyyy-MM-dd"), to_date(col("Start date"), "yyyy-MM-dd")))\
  .filter(col("Number of Participants").isNotNull() & col("Altmetric Attention Score").isNotNull() & col("HRCS HC Categories").isNotNull() & col("Duration").isNotNull()) \
  .withColumn("HRCS HC Categorie", explode(split(col("HRCS HC Categories"), "; "))) \
  .groupBy("HRCS HC Categorie") \
  .agg(
    round(avg("Duration"),2).alias("Media Durata Attesa"),
    round(avg("Number of Participants"),2).alias("Media Partecipanti"),
    round(avg("Altmetric Attention Score"),2).alias("Media Attenzione Medatica")
    ) \
  .orderBy("HRCS HC Categorie") \
  .show(truncate=False)

+----------------------------------+-------------------+------------------+-------------------------+
|HRCS HC Categorie                 |Media Durata Attesa|Media Partecipanti|Media Attenzione Medatica|
+----------------------------------+-------------------+------------------+-------------------------+
|Blood                             |1794.75            |1062.25           |42.58                    |
|Cancer                            |2326.34            |685.58            |65.32                    |
|Cardiovascular                    |1745.43            |3211.4            |67.02                    |
|Congenital                        |823.75             |324.06            |24.63                    |
|Eye                               |1069.12            |517.19            |112.14                   |
|Generic health relevance          |1340.14            |1074.57           |204.1                    |
|Infection                         |1384.75            |1274.52           |29.56  

#Analisi 9: Durata media degli studi sul cancro al seno in funzione dell’attenzione mediatica

In [18]:
df.withColumn("Duration", datediff(to_date(col("End Date"), "yyyy-MM-dd"), to_date(col("Start date"), "yyyy-MM-dd"))) \
  .withColumn("Cancer", explode(split(col("Cancer Types"), ";"))) \
  .withColumn("Cancer", trim(col("Cancer"))) \
  .filter((col("Altmetric Attention Score").isNotNull()) & (col("Number of Participants").isNotNull()) & (col("Cancer") == "Breast Cancer")) \
  .groupBy("Altmetric Attention Score") \
  .agg(
      round(avg("Duration"),2).alias("Media_Durata")
  ) \
  .orderBy(col("Altmetric Attention Score").desc()) \
  .show(30)


+-------------------------+------------+
|Altmetric Attention Score|Media_Durata|
+-------------------------+------------+
|                    784.0|      1188.0|
|                    760.0|      1324.0|
|                    680.0|      4338.0|
|                    607.0|      2471.0|
|                    506.0|      1626.0|
|                    505.0|      1891.0|
|                    349.0|      2062.0|
|                    319.0|      2261.0|
|                    318.0|      5111.0|
|                    224.0|      2223.0|
|                    191.0|      1814.0|
|                    171.0|      6232.0|
|                    148.0|      2699.0|
|                    141.0|      2387.0|
|                    137.0|      1658.0|
|                    129.0|      4383.0|
|                    117.0|      3129.0|
|                    110.0|      3289.0|
|                    107.0|      1956.0|
|                    106.0|      3684.0|
|                    104.0|      1910.0|
|               

#Analisi 10: Top 2 sponsor/collaboratori con più sperimentazioni per ogni tipo di cancro

In [20]:
from pyspark.sql.window import Window

df.withColumn("Cancer", explode(split(col("Cancer Types"), ";"))) \
    .withColumn("Cancer", trim(col("Cancer"))) \
    .withColumn("Sponsors/Collaborators", explode(split(col("Sponsors/Collaborators"), ";"))) \
    .withColumn("Sponsors/Collaborators", trim(col("Sponsors/Collaborators"))) \
    .filter(col("Cancer").isNotNull() & col("Sponsors/Collaborators").isNotNull()) \
    .groupBy("Cancer", "Sponsors/Collaborators") \
    .agg(count("*").alias("Occurrences")) \
    .withColumn("rank", row_number().over(Window.partitionBy("Cancer").orderBy(col("Occurrences").desc()))) \
    .filter(col("rank") <= 2)\
    .groupBy("Cancer") \
    .agg(
        concat_ws(", ", collect_list("Sponsors/Collaborators")).alias("Top 2 Sponsors/Collaborators"),
        concat_ws(", ", collect_list(col("Occurrences").cast("string"))).alias("Occurrences")
    )\
    .show(truncate=False)

+----------------------------------------------------------+---------------------------------------------------------------------------------------------------------------------------+-----------+
|Cancer                                                    |Top 2 Sponsors/Collaborators                                                                                               |Occurrences|
+----------------------------------------------------------+---------------------------------------------------------------------------------------------------------------------------+-----------+
|Bladder Cancer                                            |IRCCS Ospedale San Raffaele, Fondazione IRCCS Istituto Nazionale dei Tumori                                                |95, 80     |
|Bone Cancer, Osteosarcoma / Malignant Fibrous Histiocytoma|Istituto Ortopedico Rizzoli, Fondazione IRCCS Istituto Nazionale dei Tumori                                                |116, 82    |
|Brain Tumor   